In [1]:
import pandas as pd
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline
import torch

In [2]:
#อ่านไฟล์จาก test_set.csv

df = pd.read_csv('test_set.csv')
df.head(5)

,query log,status,label
0,8365-10-11T18:59:58.471006Z\t20\tQuery\tupdate...,normal,1
1,7476-01-07T01:23:53.416749Z\t43\tQuery\tGRANT ...,anormaly,0
2,7128-06-28T01:18:28.338606Z\t52\tQuery\tinsert...,normal,1
3,1467-09-24T18:12:43.073057Z\t61\tQuery\tselect...,normal,1
4,8088-02-12T20:16:22.624726Z\t93\tQuery\tupdate...,normal,1


In [4]:
#โหลด Model
path = 'Finetuned Bert Model\checkpoint-1000'
tokenizer = AutoTokenizer.from_pretrained(path)
model = AutoModelForSequenceClassification.from_pretrained(path)


# ตรวจสอบว่ามี GPU ให้ใช้งานหรือไม่ ถ้ามีให้ใช้ cuda ถ้าไม่มีให้ใช้ cpu
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

#บังคับว่าต้อง inference ที่ GPU (ถ้ามี)
model.to(device)
model.eval()

BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e

In [5]:
import re

def clean_log(text):
    # ลบ timestamp (optional แต่แนะนำ)
    text = re.sub(r"\d{4}-\d{2}-\d{2}T.*?Z", "", text)
    # แปลง tab เป็น space
    text = text.replace("\t", " ")
    return text.strip()

def predict_log(log_text):
    log_text = clean_log(log_text)
    inputs = tokenizer(
        log_text,
        return_tensors="pt",
        truncation=True,
        padding=True, # ใส่เผื่อเอาไว้ตอน inference มากกว่า 1 log (Batch Size > 1)
        max_length=128
    )
    # ⭐ ย้าย input ไป GPU
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        logits = model(**inputs).logits
        pred = torch.argmax(logits, dim=1).item()
        prob = torch.softmax(logits, dim=-1).tolist()[0]

    return "normal" if pred == 1 else "anormaly", prob

In [6]:
#วัด Accuracy

correct_predictions = 0
total_predictions = len(df)

for index, row in df.iterrows():
    text_to_classify = row['query log'] # ใช้คอลัมน์ 'query log' เป็น input
    true_label = row['status'] # ใช้คอลัมน์ 'status' เป็นคำตอบจริง (normal หรือ anomaly)

    # ทำนาย
    prediction_result,confidence = predict_log(text_to_classify)

    # ตรวจสอบว่าทำนายถูกต้องหรือไม่
    if prediction_result == true_label:
        correct_predictions += 1
        correction = 'True'
    else:
        correction = 'False'
    print(f"prediction = {prediction_result} | true_status = {true_label} | correction = {correction} | confidence = {confidence}")

prediction = normal | true_status = normal | correction = True | confidence = [2.547665462770965e-05, 0.9999744892120361]
prediction = anormaly | true_status = anormaly | correction = True | confidence = [0.9999727010726929, 2.7352072720532306e-05]
prediction = normal | true_status = normal | correction = True | confidence = [2.9086269933031872e-05, 0.9999709129333496]
prediction = normal | true_status = normal | correction = True | confidence = [2.7102600142825395e-05, 0.999972939491272]
prediction = normal | true_status = normal | correction = True | confidence = [2.57483607128961e-05, 0.999974250793457]
prediction = normal | true_status = normal | correction = True | confidence = [2.5770370484679006e-05, 0.999974250793457]
prediction = anormaly | true_status = anormaly | correction = True | confidence = [0.9999716281890869, 2.8385476980474778e-05]
prediction = normal | true_status = normal | correction = True | confidence = [2.750370003923308e-05, 0.9999724626541138]
prediction = no

KeyboardInterrupt: 

In [6]:
# วัด Accuracy
accuracy = (correct_predictions / total_predictions) * 100
print(f"test_set จำนวน {total_predictions}")
print(f"ทำนายถูกจำนวน {correct_predictions}")
print(f"Accuracy = {accuracy}")

test_set จำนวน 499500
ทำนายถูกจำนวน 454528
Accuracy = 90.9965965965966
